This notebook demonstrates how to use [ESM-2](https://github.com/facebookresearch/esm) transformer protein language model to extract embeddings from provided protein sequences. The following code calculates the embeddings for each protein sequence in the training and test FASTA files and saves them as individual `.pt` files. 

The resulting embedding can be loaded using the following code:
```
import torch

embedding = torch.load('[EntryID].pt')
embedding = embedding['mean_representations'][33].numpy()
```

Computing the embeddings and subsequently reading in the `.pt` files can take a while. The resulting numpy arrays can be found [here](https://www.kaggle.com/datasets/viktorfairuschin/cafa-5-ems-2-embeddings-numpy).

**Note** that the test FASTA file contains duplicate entries. For this reason, this notebook uses cleaned FASTA files, which can be found [here](https://www.kaggle.com/datasets/viktorfairuschin/cafa-5-fasta-files).

In [1]:
!pip install -q fair-esm

In [2]:
from tqdm.auto import tqdm
from esm import FastaBatchedDataset, pretrained
import joblib
import pathlib
import torch
from tqdm.notebook import tqdm

In [3]:
def extract_embeddings(model_name, fasta_file, output_dir, tokens_per_batch=4096, seq_length=1022, repr_layers=[33]):
    import time
    
    model, alphabet = pretrained.load_model_and_alphabet(model_name)
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
        
    dataset = FastaBatchedDataset.from_file(fasta_file)
    batches = dataset.get_batch_indices(tokens_per_batch, extra_toks_per_seq=1)
    data_loader = torch.utils.data.DataLoader(
        dataset, 
        collate_fn=alphabet.get_batch_converter(seq_length), 
        batch_sampler=batches
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    embedding_res = {}
    
    # Calculate total batches for progress tracking
    total_batches = len(batches)
    print(f"Starting extraction: {total_batches} batches to process")
    print("-" * 50)
    
    start_time = time.time()
    
    with torch.no_grad():
        for batch_idx, (labels, strs, toks) in enumerate(data_loader):
            # Progress tracking
            if batch_idx % 100 == 0 or batch_idx == total_batches - 1:
                elapsed_time = time.time() - start_time
                progress = (batch_idx + 1) / total_batches
                eta = (elapsed_time / (batch_idx + 1)) * (total_batches - batch_idx - 1)
                
                print(f"Batch {batch_idx + 1}/{total_batches} | "
                      f"Progress: {progress*100:.1f}% | "
                      f"Elapsed: {elapsed_time:.1f}s | "
                      f"ETA: {eta:.1f}s")
            
            toks = toks.to(device="cuda", non_blocking=True)
            out = model(toks, repr_layers=repr_layers, return_contacts=False)
            representations = {layer: t.to(device="cpu") for layer, t in out["representations"].items()}
            
            for i, label in enumerate(labels):
                entry_id = label.split()[0]
                truncate_len = min(seq_length, len(strs[i]))
                
                # Get the embedding from the last layer
                final_layer = max(repr_layers)
                embedding_res[entry_id] = representations[final_layer][i, 1 : truncate_len + 1].mean(0).clone().numpy()
    
    total_time = time.time() - start_time
    print("-" * 50)
    print(f"✓ Extraction complete!")
    print(f"  Total time: {total_time:.2f} seconds")
    print(f"  Proteins processed: {len(embedding_res)}")
    print(f"  Average time per batch: {total_time/total_batches:.2f}s")
    
    joblib.dump(embedding_res, 'embedding_test.joblib')
    print(f"✓ Saved embeddings to embedding_test.joblib")
    

model_name = 'esm2_t33_650M_UR50D'
fasta_file = pathlib.Path('/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset.fasta')
output_dir = pathlib.Path('test_embeddings')
extract_embeddings(model_name, fasta_file, output_dir)

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt


Starting extraction: 26079 batches to process
--------------------------------------------------
Batch 1/26079 | Progress: 0.0% | Elapsed: 0.1s | ETA: 1874.6s
Batch 101/26079 | Progress: 0.4% | Elapsed: 90.5s | ETA: 23268.3s
Batch 201/26079 | Progress: 0.8% | Elapsed: 176.8s | ETA: 22768.0s
Batch 301/26079 | Progress: 1.2% | Elapsed: 262.7s | ETA: 22499.6s
Batch 401/26079 | Progress: 1.5% | Elapsed: 348.5s | ETA: 22314.3s
Batch 501/26079 | Progress: 1.9% | Elapsed: 434.1s | ETA: 22164.2s
Batch 601/26079 | Progress: 2.3% | Elapsed: 519.9s | ETA: 22039.1s
Batch 701/26079 | Progress: 2.7% | Elapsed: 606.4s | ETA: 21954.9s
Batch 801/26079 | Progress: 3.1% | Elapsed: 693.5s | ETA: 21886.2s
Batch 901/26079 | Progress: 3.5% | Elapsed: 780.6s | ETA: 21812.6s
Batch 1001/26079 | Progress: 3.8% | Elapsed: 867.3s | ETA: 21728.7s
Batch 1101/26079 | Progress: 4.2% | Elapsed: 954.0s | ETA: 21643.2s
Batch 1201/26079 | Progress: 4.6% | Elapsed: 1040.9s | ETA: 21561.2s
Batch 1301/26079 | Progress: 5.0% 

## Process train file

## Process test file